## 1. Install dependencies


In [1]:
!pip install -q transformers accelerate pandas


## 2. Upload your database

Run this cell, then use the file picker to upload `company.db` (or any SQLite `.db` file).


In [2]:
from google.colab import files

uploaded = files.upload()
DB_PATH = list(uploaded.keys())[0]
print(f"Uploaded: {DB_PATH}")


Saving company.db to company.db
Uploaded: company.db


## 3. Load the model

Using **Qwen2.5-Coder-3B-Instruct** — a small, free, permissively-licensed model tuned for code/SQL generation. It fits comfortably on Colab's free T4 GPU and downloads in about a minute.

If you want higher accuracy and don't mind a slower load, swap `MODEL_NAME` below for `"Qwen/Qwen2.5-Coder-7B-Instruct"` (still fits on a T4, just slower).


In [3]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_NAME = "Qwen/Qwen2.5-Coder-3B-Instruct"

device = "cuda" if torch.cuda.is_available() else "cpu"
if device == "cpu":
    print("WARNING: no GPU detected — this will be slow. Go to Runtime > Change runtime type > T4 GPU.")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16 if device == "cuda" else torch.float32,
    device_map="auto",
)
print(f"Loaded {MODEL_NAME} on {device}")


config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

Loaded Qwen/Qwen2.5-Coder-3B-Instruct on cuda


## 4. Connect to the database


In [4]:
import sqlite3
import pandas as pd

conn = sqlite3.connect(DB_PATH)

def get_schema(conn):
    cur = conn.cursor()
    cur.execute("SELECT name FROM sqlite_master WHERE type='table' AND name NOT LIKE 'sqlite_%' ORDER BY name")
    tables = [r[0] for r in cur.fetchall()]
    lines = []
    for t in tables:
        cur.execute(f"PRAGMA table_info({t})")
        cols = [row[1] for row in cur.fetchall()]
        lines.append(f"{t}({', '.join(cols)})")
    return "\n".join(lines), tables

schema_text, tables = get_schema(conn)
print(f"Loaded {DB_PATH} — {len(tables)} tables:\n")
print(schema_text)


Loaded company.db — 8 tables:

clients(client_id, company_name, industry, contact_name, contact_email, country)
departments(department_id, name, office_id, budget)
employees(employee_id, first_name, last_name, email, job_title, department_id, manager_id, hire_date, office_id, status)
offices(office_id, city, country)
project_assignments(assignment_id, project_id, employee_id, role, allocation_pct)
projects(project_id, name, client_id, department_id, start_date, end_date, budget, status)
salaries(salary_id, employee_id, annual_salary, currency, effective_date)
timesheets(timesheet_id, employee_id, project_id, work_date, hours)


## 5. Define the agent

`ask(question)` builds a prompt with the schema, runs local inference, extracts the SQL, executes it, and returns a DataFrame. Keeps a short rolling history so follow-ups work.


In [5]:
_history = []
MAX_NEW_TOKENS = 200

SYSTEM_PROMPT_TEMPLATE = """You are a SQLite expert. Given the database schema below, write exactly ONE valid SQLite SELECT query that answers the user's question.

Schema:
{schema}

Rules:
- Output ONLY the raw SQL statement, nothing else.
- No markdown code fences, no explanation, no commentary, no preamble.
- Use only tables/columns that exist in the schema above.
- End the statement with a semicolon.
- Prefer readable aliases for computed columns.
- If the question refers to a previous question (e.g. \"now filter by X\"), use conversation history for context."""

def extract_sql(text):
    t = text.strip()
    if "```" in t:
        parts = t.split("```")
        if len(parts) >= 2:
            t = parts[1]
            if t.lower().startswith("sql"):
                t = t[3:]
    t = t.strip()
    if ";" in t:
        t = t.split(";")[0] + ";"
    return t.strip()

def _generate(question):
    system_prompt = SYSTEM_PROMPT_TEMPLATE.format(schema=schema_text)
    messages = [{"role": "system", "content": system_prompt}] + _history + [{"role": "user", "content": question}]

    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            temperature=1.0,
            pad_token_id=tokenizer.eos_token_id,
        )

    new_tokens = output_ids[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True)

def ask(question, show_sql=True):
    """Ask a natural-language question, run the generated SQL, return a DataFrame."""
    raw = _generate(question)
    sql = extract_sql(raw)
    if show_sql:
        print(sql, "\n")
    try:
        df = pd.read_sql_query(sql, conn)
    except Exception as e:
        print(f"SQL error: {e}")
        print("(try rephrasing your question)")
        return None
    _history.append({"role": "user", "content": question})
    _history.append({"role": "assistant", "content": sql})
    del _history[:-10]
    return df


## 6. Ask questions


In [11]:
ask("Which department has the highest average salary?", show_sql=False)

,department_name,average_salary
0,Product,126535.714286


In [13]:
ask("List the 5 most expensive active projects" ,show_sql=False)

,project_name,budget


In [14]:
# Follow-up questions can reference the previous one
ask("Now show only the ones with a budget over 200000",show_sql=False)


,project_name,budget


In [15]:
ask("List the top 5 projects with the highest budget", show_sql=False)

,project_name,budget
0,Initech Automation,490000.0
1,Soylent Expansion,487000.0
2,Cyberdyne Analytics Suite,473000.0
3,Stark Migration,471000.0
4,Hooli Optimization,456000.0
